# DI 725 Term Project - Phase 3
**Title**: Multi-Modal Transformers for Remote Sensing Segmentation
**Phase 3**: Ablation, Discussion, Conclusion

This notebook contains the finalized codebase for Phase 3. It includes:
1. Configuration and Setup
2. Dataset and Dataloading
3. Model Architectures (Baseline & Multi-Modal)
4. Training and Evaluation Routines
5. Main Experiments
6. Ablation Study


In [ ]:
import os, random, re, warnings
import wandb
wandb.login(key="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")
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.optim.lr_scheduler import CosineAnnealingLR

warnings.filterwarnings('ignore')


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.


AuthenticationError: API key must have 40+ characters, has 36.

## 1. Configuration & Setup
Grouping all hyperparameters and paths to avoid magic numbers.


In [ ]:
class CFG:
    seed = 42
    
    # Paths
    data_dir = '../DI725_project_dataset'
    img_dir = os.path.join(data_dir, 'images')
    msk_dir = os.path.join(data_dir, 'masks')
    cap_path = os.path.join(data_dir, 'captions.csv')
    
    # Model parameters
    img_size = 128
    patch_size = 16
    max_len = 50
    dim = 192
    depth = 5
    heads = 6
    text_dim = 192
    text_depth = 3
    
    # Training parameters
    batch_size = 16
    epochs_baseline = 30
    epochs_mm = 50
    lr = 3e-4
    weight_decay = 1e-4
    
    # Ablation parameters
    epochs_ablation = 10
    
    # Default Caption Column
    caption_col = 'text_qwen3-4b'

# Set random seeds for reproducibility
random.seed(CFG.seed)
np.random.seed(CFG.seed)
torch.manual_seed(CFG.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG.seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


## 2. Dataset Preparation
Class definitions and visual mappings.


In [ ]:
# Class definitions
COLOR_MAP = [
    ('Tree',     (0,   100, 0  )),
    ('Shrub',    (255, 182, 193)),
    ('Grass',    (154, 205, 50 )),
    ('Crop',     (255, 215, 0  )),
    ('Built-up', (139, 69,  19 )),
    ('Barren',   (211, 211, 211)),
    ('Water',    (0,   0,   255)),
]
CLASS_NAMES = [c[0] for c in COLOR_MAP]
CLASS_COLORS = [c[1] for c in COLOR_MAP]
NUM_CLASSES = len(CLASS_NAMES)
RGB_TO_IDX = {rgb: i for i, (_, rgb) in enumerate(COLOR_MAP)}

print(f'{NUM_CLASSES} classes: {CLASS_NAMES}')


In [ ]:
# Load Metadata
df = pd.read_csv(CFG.cap_path)
print(f'Dataset size: {len(df)} images')

def rgb_to_mask(mask_img):
    """Convert RGB mask image to class index array."""
    mask = np.array(mask_img)
    out = np.zeros((mask.shape[0], mask.shape[1]), dtype=np.uint8)
    for rgb, idx in RGB_TO_IDX.items():
        out[(mask == rgb).all(axis=-1)] = idx
    return out


### Tokenizer


In [ ]:
def simple_tokenize(text):
    return re.findall(r"[a-zA-Z]+|\\d+", str(text).lower())

class SimpleTokenizer:
    """Minimal word-level tokenizer."""
    def __init__(self, texts, max_vocab=4000, min_freq=2):
        freq = {}
        for t in texts:
            for tok in simple_tokenize(t):
                freq[tok] = freq.get(tok, 0) + 1
        vocab = [tok for tok, f in sorted(freq.items(), key=lambda x: -x[1])
                 if f >= min_freq][:max_vocab]
        self.itos = ['[PAD]', '[UNK]'] + vocab
        self.stoi = {t: i for i, t in enumerate(self.itos)}
        self.pad_idx = 0

    def encode(self, text, max_len=CFG.max_len):
        toks = simple_tokenize(text)
        ids = [self.stoi.get(t, 1) for t in toks][:max_len]
        ids += [0] * (max_len - len(ids))
        return np.array(ids, dtype=np.int64)

    def __len__(self):
        return len(self.itos)

tokenizer = SimpleTokenizer(df[CFG.caption_col].tolist())
print(f'Vocabulary size: {len(tokenizer)}')


### Dataset and DataLoader


In [ ]:
class SegDataset(Dataset):
    def __init__(self, dataframe, img_dir, msk_dir, tokenizer,
                 caption_col=CFG.caption_col, img_size=CFG.img_size,
                 max_len=CFG.max_len, augment=False):
        self.df = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.msk_dir = msk_dir
        self.tokenizer = tokenizer
        self.caption_col = caption_col
        self.max_len = max_len
        self.img_size = img_size
        self.augment = augment
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225])
        self.color_jitter = transforms.ColorJitter(
            brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        fname = row['filename']

        img = Image.open(os.path.join(self.img_dir, fname)).convert('RGB')
        img = img.resize((self.img_size, self.img_size), Image.BILINEAR)

        msk = Image.open(os.path.join(self.msk_dir, fname)).convert('RGB')
        msk = msk.resize((self.img_size, self.img_size), Image.NEAREST)

        if self.augment:
            if random.random() > 0.5:
                img = img.transpose(Image.FLIP_LEFT_RIGHT)
                msk = msk.transpose(Image.FLIP_LEFT_RIGHT)
            if random.random() > 0.5:
                img = img.transpose(Image.FLIP_TOP_BOTTOM)
                msk = msk.transpose(Image.FLIP_TOP_BOTTOM)
            k = random.randint(0, 3)
            if k > 0:
                img = img.rotate(k * 90, expand=False)
                msk = msk.rotate(k * 90, expand=False)

        img = transforms.ToTensor()(img)
        if self.augment:
            img = self.color_jitter(img)
        img = self.normalize(img)

        msk = torch.from_numpy(rgb_to_mask(msk)).long()
        tok = torch.from_numpy(
            self.tokenizer.encode(row[self.caption_col], self.max_len))

        return img, msk, tok


In [ ]:
# Full dataset split
all_idx = np.arange(len(df))
np.random.shuffle(all_idx)

split = int(0.8 * len(df))
train_df = df.iloc[all_idx[:split]]
val_df   = df.iloc[all_idx[split:]]

train_ds = SegDataset(train_df, CFG.img_dir, CFG.msk_dir, tokenizer, augment=True)
val_ds   = SegDataset(val_df,   CFG.img_dir, CFG.msk_dir, tokenizer, augment=False)

train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=CFG.batch_size, shuffle=False, num_workers=0)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')


In [ ]:
# Compute class weights based on composition
comp_cols = ['Tree', 'Shrub', 'Grass', 'Crop', 'Built-up', 'Barren', 'Water']
avg = train_df[comp_cols].mean().values + 1e-6
inv_freq = 1.0 / avg
class_weights = inv_freq / inv_freq.sum() * NUM_CLASSES
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

print("Class Weights:")
for name, w in zip(CLASS_NAMES, class_weights):
    print(f'  {name:10s}: {w:.3f}')


## 3. Models
### 3.1 Baseline: Image-Only Transformer


In [ ]:
class ImageSegTransformer(nn.Module):
    """Baseline image-only segmentation transformer."""
    def __init__(self, img_size=CFG.img_size, patch_size=CFG.patch_size, dim=CFG.dim,
                 depth=CFG.depth, heads=CFG.heads, num_classes=NUM_CLASSES):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2

        self.patch_embed = nn.Conv2d(3, dim, kernel_size=patch_size, stride=patch_size)
        self.pos_embed = nn.Parameter(torch.randn(1, self.num_patches, dim) * 0.02)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=dim, nhead=heads, dim_feedforward=dim * 4,
            dropout=0.1, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=depth)

        self.head = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, num_classes)
        )

    def forward(self, x):
        B = x.size(0)
        x = self.patch_embed(x)
        H, W = x.shape[-2:]
        x = x.flatten(2).transpose(1, 2)
        x = x + self.pos_embed
        x = self.encoder(x)
        logits = self.head(x)
        logits = logits.transpose(1, 2).reshape(B, -1, H, W)
        logits = F.interpolate(logits, size=(self.img_size, self.img_size),
                               mode='bilinear', align_corners=False)
        return logits


### 3.2 Multi-Modal Transformer


In [ ]:
class ImageTextSegTransformer(nn.Module):
    """Multi-modal segmentation transformer."""
    def __init__(self, vocab_size, max_len=CFG.max_len,
                 img_size=CFG.img_size, patch_size=CFG.patch_size,
                 dim=CFG.dim, depth=CFG.depth, heads=CFG.heads,
                 text_dim=CFG.text_dim, text_depth=CFG.text_depth,
                 num_classes=NUM_CLASSES):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        num_patches = (img_size // patch_size) ** 2

        self.patch_embed = nn.Conv2d(3, dim, kernel_size=patch_size, stride=patch_size)
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches, dim) * 0.02)
        img_enc_layer = nn.TransformerEncoderLayer(
            d_model=dim, nhead=heads, dim_feedforward=dim * 4,
            dropout=0.1, batch_first=True)
        self.img_encoder = nn.TransformerEncoder(img_enc_layer, num_layers=depth)

        self.text_embed = nn.Embedding(vocab_size, text_dim, padding_idx=0)
        self.text_pos = nn.Parameter(torch.randn(1, max_len, text_dim) * 0.02)
        txt_enc_layer = nn.TransformerEncoderLayer(
            d_model=text_dim, nhead=heads, dim_feedforward=text_dim * 4,
            dropout=0.1, batch_first=True)
        self.text_encoder = nn.TransformerEncoder(txt_enc_layer, num_layers=text_depth)
        self.text_proj = nn.Linear(text_dim, dim)

        self.fuse = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.GELU(),
            nn.LayerNorm(dim)
        )

        self.head = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, num_classes)
        )

    def forward(self, x, tokens):
        B = x.size(0)

        # Image features
        x = self.patch_embed(x)
        H, W = x.shape[-2:]
        x = x.flatten(2).transpose(1, 2)
        x = x + self.pos_embed
        x = self.img_encoder(x)

        # Text features
        pad_mask = (tokens == 0)
        t = self.text_embed(tokens) + self.text_pos
        t = self.text_encoder(t, src_key_padding_mask=pad_mask)
        mask_f = (~pad_mask).float().unsqueeze(-1)
        t_pool = (t * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1)
        t_proj = self.text_proj(t_pool)

        # Fusion
        t_expand = t_proj.unsqueeze(1).expand(-1, x.size(1), -1)
        fused = torch.cat([x, t_expand], dim=-1)
        fused = self.fuse(fused)

        logits = self.head(fused)
        logits = logits.transpose(1, 2).reshape(B, -1, H, W)
        logits = F.interpolate(logits, size=(self.img_size, self.img_size),
                               mode='bilinear', align_corners=False)
        return logits


## 4. Training & Evaluation Utilities


In [ ]:
def compute_iou_per_class(pred, target, num_classes=NUM_CLASSES):
    pred = pred.view(-1)
    target = target.view(-1)
    ious = {}
    for c in range(num_classes):
        p = pred == c
        t = target == c
        inter = (p & t).sum().item()
        union = (p | t).sum().item()
        if union > 0:
            ious[c] = inter / union
    return ious

def mean_iou(pred, target, num_classes=NUM_CLASSES):
    ious = compute_iou_per_class(pred, target, num_classes)
    return float(np.mean(list(ious.values()))) if ious else 0.0

def train_one_epoch(model, loader, optimizer, weights, use_text=False):
    model.train()
    total_loss, n = 0.0, 0
    for img, msk, tok in loader:
        img, msk, tok = img.to(device), msk.to(device), tok.to(device)
        logits = model(img, tok) if use_text else model(img)
        loss = F.cross_entropy(logits, msk, weight=weights)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * img.size(0)
        n += img.size(0)
    return total_loss / n

@torch.no_grad()
def evaluate(model, loader, weights, use_text=False):
    model.eval()
    total_loss, total_iou, n = 0.0, 0.0, 0
    per_class_ious = {c: [] for c in range(NUM_CLASSES)}
    for img, msk, tok in loader:
        img, msk, tok = img.to(device), msk.to(device), tok.to(device)
        logits = model(img, tok) if use_text else model(img)
        loss = F.cross_entropy(logits, msk, weight=weights)
        pred = logits.argmax(dim=1)
        
        total_loss += loss.item() * img.size(0)
        total_iou += mean_iou(pred.cpu(), msk.cpu()) * img.size(0)
        
        cls_ious = compute_iou_per_class(pred.cpu(), msk.cpu())
        for c, v in cls_ious.items():
            per_class_ious[c].append(v)
        n += img.size(0)
        
    avg_class = {c: np.mean(v) if v else 0.0 for c, v in per_class_ious.items()}
    return total_loss / n, total_iou / n, avg_class


## 5. Main Experiments
### 5.1 Train Baseline


In [ ]:
baseline = ImageSegTransformer().to(device)
opt_base = torch.optim.AdamW(baseline.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
sched_base = CosineAnnealingLR(opt_base, T_max=CFG.epochs_baseline, eta_min=1e-6)

# Initialize W&B
run_base = wandb.init(
    project="GeoCaption-Seg",
    entity="kaan-kocaturk99-odt-",
    name="Baseline_Image_Only",
    config={
        "seed": CFG.seed,
        "img_size": CFG.img_size,
        "patch_size": CFG.patch_size,
        "dim": CFG.dim,
        "depth": CFG.depth,
        "heads": CFG.heads,
        "batch_size": CFG.batch_size,
        "epochs": CFG.epochs_baseline,
        "lr": CFG.lr,
        "weight_decay": CFG.weight_decay,
        "model_type": "baseline"
    }
)

history_base = {'train_loss': [], 'val_loss': [], 'val_miou': []}
best_base_miou = 0.0
best_base_state = None

print("Training Baseline...")
for epoch in range(1, CFG.epochs_baseline + 1):
    tr_loss = train_one_epoch(baseline, train_loader, opt_base, class_weights, use_text=False)
    sched_base.step()
    va_loss, va_miou, va_cls = evaluate(baseline, val_loader, class_weights, use_text=False)
    
    history_base['train_loss'].append(tr_loss)
    history_base['val_loss'].append(va_loss)
    history_base['val_miou'].append(va_miou)

    if va_miou > best_base_miou:
        best_base_miou = va_miou
        best_base_state = {k: v.clone() for k, v in baseline.state_dict().items()}

    print(f'[Baseline] Epoch {epoch:2d}/{CFG.epochs_baseline} '
          f'train_loss={tr_loss:.4f} val_mIoU={va_miou:.4f}')
    
    # Log to W&B
    log_dict = {
        "epoch": epoch,
        "train_loss": tr_loss,
        "val_loss": va_loss,
        "val_miou": va_miou
    }
    for c, v in va_cls.items():
        log_dict[f"val_iou_{CLASS_NAMES[c]}"] = v
    wandb.log(log_dict)

if best_base_state:
    baseline.load_state_dict(best_base_state)

# Finish W&B run
run_base.finish()


### 5.2 Train Multi-Modal


In [ ]:
multimodal = ImageTextSegTransformer(vocab_size=len(tokenizer)).to(device)
opt_mm = torch.optim.AdamW(multimodal.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
sched_mm = CosineAnnealingLR(opt_mm, T_max=CFG.epochs_mm, eta_min=1e-6)

# Initialize W&B
run_mm = wandb.init(
    project="GeoCaption-Seg",
    entity="kaan-kocaturk99-odt-",
    name="MultiModal",
    config={
        "seed": CFG.seed,
        "img_size": CFG.img_size,
        "patch_size": CFG.patch_size,
        "dim": CFG.dim,
        "depth": CFG.depth,
        "heads": CFG.heads,
        "text_dim": CFG.text_dim,
        "text_depth": CFG.text_depth,
        "batch_size": CFG.batch_size,
        "epochs": CFG.epochs_mm,
        "lr": CFG.lr,
        "weight_decay": CFG.weight_decay,
        "caption_col": CFG.caption_col,
        "model_type": "multimodal"
    }
)

history_mm = {'train_loss': [], 'val_loss': [], 'val_miou': []}
best_mm_miou = 0.0
best_mm_state = None
best_mm_cls = {}

print("Training Multi-Modal...")
for epoch in range(1, CFG.epochs_mm + 1):
    tr_loss = train_one_epoch(multimodal, train_loader, opt_mm, class_weights, use_text=True)
    sched_mm.step()
    va_loss, va_miou, va_cls = evaluate(multimodal, val_loader, class_weights, use_text=True)
    
    history_mm['train_loss'].append(tr_loss)
    history_mm['val_loss'].append(va_loss)
    history_mm['val_miou'].append(va_miou)

    if va_miou > best_mm_miou:
        best_mm_miou = va_miou
        best_mm_cls = va_cls.copy()
        best_mm_state = {k: v.clone() for k, v in multimodal.state_dict().items()}

    print(f'[MultiModal] Epoch {epoch:2d}/{CFG.epochs_mm} '
          f'train_loss={tr_loss:.4f} val_mIoU={va_miou:.4f}')
    
    # Log to W&B
    log_dict = {
        "epoch": epoch,
        "train_loss": tr_loss,
        "val_loss": va_loss,
        "val_miou": va_miou
    }
    for c, v in va_cls.items():
        log_dict[f"val_iou_{CLASS_NAMES[c]}"] = v
    wandb.log(log_dict)

if best_mm_state:
    multimodal.load_state_dict(best_mm_state)

# Finish W&B run
run_mm.finish()


### 5.3 Results Visualization


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_base['val_loss'], 'b-', label='Baseline val')
axes[0].plot(history_mm['val_loss'], 'r-', label='Multi-modal val')
axes[0].set_title('Validation Loss')
axes[0].legend()

axes[1].plot(history_base['val_miou'], 'b-o', label='Baseline')
axes[1].plot(history_mm['val_miou'], 'r-o', label='Multi-modal')
axes[1].set_title('Validation mIoU')
axes[1].legend()
plt.show()

# Reevaluate both to get final per-class IoU
_, _, base_cls = evaluate(baseline, val_loader, class_weights, use_text=False)
_, _, mm_cls = evaluate(multimodal, val_loader, class_weights, use_text=True)

x = np.arange(NUM_CLASSES)
w = 0.35
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - w/2, [base_cls.get(c, 0) for c in range(NUM_CLASSES)], w, label='Baseline', color='steelblue')
ax.bar(x + w/2, [mm_cls.get(c, 0) for c in range(NUM_CLASSES)], w, label='Multi-modal', color='coral')
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES)
ax.set_ylabel('IoU')
ax.set_title('Per-Class IoU Comparison')
ax.legend()
plt.show()


## 6. Ablation Study: Caption Source Impact
Evaluating different textual foundations.


In [ ]:
caption_columns = [
    'hybrid_gemma3-4b',
    'hybrid_qwen3-vl-8b',
    'text_qwen3-4b',
    'vision_gemma3-4b',
    'vision_qwen3-vl-8b',
]

ablation_results = {}

for cap_col in caption_columns:
    print(f'\n--- Training with caption: {cap_col} ---')
    
    # Initialize W&B for ablation run
    run_abl = wandb.init(
        project="GeoCaption-Seg",
        entity="kaan-kocaturk99-odt-",
        name=f"Ablation_{cap_col}",
        config={
            "seed": CFG.seed,
            "img_size": CFG.img_size,
            "patch_size": CFG.patch_size,
            "dim": CFG.dim,
            "depth": CFG.depth,
            "heads": CFG.heads,
            "text_dim": CFG.text_dim,
            "text_depth": CFG.text_depth,
            "batch_size": CFG.batch_size,
            "epochs": CFG.epochs_ablation,
            "lr": CFG.lr,
            "weight_decay": CFG.weight_decay,
            "caption_col": cap_col,
            "model_type": "multimodal_ablation"
        }
    )
    
    # Reload datasets with new caption column
    train_ds_abl = SegDataset(train_df, CFG.img_dir, CFG.msk_dir, tokenizer, caption_col=cap_col, augment=True)
    val_ds_abl = SegDataset(val_df, CFG.img_dir, CFG.msk_dir, tokenizer, caption_col=cap_col, augment=False)
    train_loader_abl = DataLoader(train_ds_abl, batch_size=CFG.batch_size, shuffle=True)
    val_loader_abl = DataLoader(val_ds_abl, batch_size=CFG.batch_size)

    model_abl = ImageTextSegTransformer(vocab_size=len(tokenizer)).to(device)
    opt_abl = torch.optim.AdamW(model_abl.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
    sched_abl = CosineAnnealingLR(opt_abl, T_max=CFG.epochs_ablation, eta_min=1e-5)

    best_miou = 0.0
    
    for epoch in range(1, CFG.epochs_ablation + 1):
        tr_loss = train_one_epoch(model_abl, train_loader_abl, opt_abl, class_weights, use_text=True)
        sched_abl.step()
        va_loss, va_miou, va_cls = evaluate(model_abl, val_loader_abl, class_weights, use_text=True)
        if va_miou > best_miou:
            best_miou = va_miou
        
        # Log to W&B
        log_dict = {
            "epoch": epoch,
            "train_loss": tr_loss,
            "val_loss": va_loss,
            "val_miou": va_miou
        }
        for c, v in va_cls.items():
            log_dict[f"val_iou_{CLASS_NAMES[c]}"] = v
        wandb.log(log_dict)

    ablation_results[cap_col] = best_miou
    print(f'✓ Best mIoU for {cap_col}: {best_miou:.4f}')

    # Finish W&B run
    run_abl.finish()

    del model_abl, opt_abl, sched_abl
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
# Ablation Visualization
fig, ax = plt.subplots(figsize=(10, 4))
caps = list(ablation_results.keys())
mious = [ablation_results[c] for c in caps]
ax.barh(caps, mious, color='mediumpurple')
ax.set_xlabel('mIoU')
ax.set_title('Ablation Study: Impact of Caption Generation Model')
for i, v in enumerate(mious):
    ax.text(v + 0.005, i, f'{v:.4f}', va='center')
plt.tight_layout()
plt.show()
